# 🍌 Nano Banana Studio & API Engine (موتور و پنل تحت وب تولید تصویر)

کنترل پنل تحت وب و سرور API برای ساخت رایگان و پرسرعت تصویر با کیفیت 1024x1024 روی گرافیک گوگل کلب (T4 GPU)، متصل به مینیس.

### 🚀 امکانات:
- 🎨 **پنل تحت وب کامل**: استودیو ساخت تصویر، گالری آنلاین، و تولید کلید API
- 🔑 **سازگاری ۱۰۰٪ با فرمت OpenAI API** ()
- 🌐 **تانل پرسرعت Cloudflare**: بدون نیاز به فیلترشکن برای مینیس
- ⚡ **تولید در کمتر از ۲ ثانیه** بدون هزینه و بدون لیمیت توکن

---

In [ ]:
#@title 1. بررسی وضعیت کارت گرافیک رایگان (GPU Check)
!nvidia-smi


In [ ]:
# 2. نصب پکیج‌ها و کلودفلرد
!pip install -q fastapi uvicorn "diffusers>=0.28.0" transformers accelerate pycloudflared pillow requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ پیش‌نیازها آماده شدند!")


In [ ]:
# 3. دریافت آخرین نسخه پنل و سرور از گیت‌هاب
!wget -q -O server.py https://raw.githubusercontent.com/3krbkbkrbrbg/nano-banana-colab/main/server.py
print("✅ سرور و پنل دریافت شد.")


In [ ]:
# 4. روشن کردن سرور و ایجاد تانل امن کلودفلر
import subprocess, time, re

!pkill -f "uvicorn" || true
!pkill -f "cloudflared" || true

print("🚀 در حال استارت سرور و پنل روی پورت 8000...")
server_proc = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

print("🌐 در حال فعال‌سازی تانل کلودفلر...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("
" + "="*60)
    print("🎉 پنل و API آماده استفاده است!")
    print(f"🌐 آدرس پنل تحت وب (استودیو + گالری): {public_url}")
    print(f"🔑 آدرس API برای مینیس: {public_url}/v1")
    print(f"🛡️ کلید پیش‌فرض API: sk-nanobanana-free")
    print("="*60)
    print("
📌 دستور اتصال مستقیم در مینیس:")
    print(f"minis-config set providers.65401b6b-7d1f-44ad-bbe4-00e1f4d654de.customBaseURL "{public_url}/v1"")
else:
    print("❌ خطا در ساخت تانل. سلول را مجدداً اجرا کنید.")


In [ ]:
# 5. روشن نگه‌داشتن دائمی سشن در پس‌زمینه (Anti-Sleep & Keep-Alive)
from IPython.display import display, HTML
import time

# پخش صدای بی‌صدا در پس‌زمینه برای جلوگیری از Sleep شدن تب در موبایل و مرورگر
keepalive_html = """
<div style="padding: 10px; background: #1e293b; color: #38bdf8; border-radius: 8px; font-family: monospace;">
  🟢 سنسور Keep-Alive فعال شد. تب در پس‌زمینه زنده می‌ماند.
</div>
<audio loop autoplay style="display:none;">
  <source src="data:audio/wav;base64,UklGRigAAABXQVZFZm10IBAAAAABAAEARKwAAIhYAQACABAAZGF0YQQAAAAAAA==" type="audio/wav">
</audio>
<script>
  function keepAlive() {
    console.log("[Keep-Alive] Ping");
    var btn = document.querySelector("colab-connect-button");
    if(btn && btn.shadowRoot) {
      var inner = btn.shadowRoot.querySelector("#connect");
      if(inner) inner.click();
    }
  }
  setInterval(keepAlive, 45000);
</script>
"""
display(HTML(keepalive_html))

print("🚀 سرور بدون قطعی در حال اجراست...")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("متوقف شد.")
